# Day 7: free vision model on Colab ($0)

Runs the open-source **Qwen2.5-VL 7B** vision model through Ollama on Colab's free GPU and reads the same Carondelet (1938) tiles Tesseract was tested on. Nothing here calls a paid API.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all (about 10 minutes; most of it is the 6 GB model download).

The repo must be pushed to GitHub first so Colab can clone it.

In [ ]:
# 1. Install Tesseract and Ollama
!apt-get -qq update && apt-get -qq install -y tesseract-ocr zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

In [ ]:
# 2. Start the Ollama server in the background and download the model (~6 GB)
import subprocess
import time

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen2.5vl:7b

In [ ]:
# 3. Get the project code
![ -d BlockHistory ] || git clone -q https://github.com/Hariswar/BlockHistory.git
%cd /content/BlockHistory
!git pull -q
!pip -q install -e ".[vision]"

In [ ]:
# 4. Download and tile the two test sheets (map images aren't in git)
!python scripts/download_sheet.py carondelet 04858_12_1938-0050
!python scripts/tile_sheet.py data/sheets/carondelet/1938/*.jpg

## 5. Read the tiles
Each run prints Tesseract's words, the vision model's buildings, and for both: which of your hand labels they found or MISSED, plus a gas-tank count. All three tiles are from sheet 50, whose labels are positioned by tile.
- `0050_r3_c2`: **BULK OIL STA.** (gasoline storage tanks), ice plant, vinegar works, wrecking yard, Semi-Steel Casting
- `0050_r3_c0`: **FILL'G STA.** with gas tanks, gasoline & alcohol storage, coal yard
- `0050_r1_c2`: **OIL HO.**, locomotive house

Tesseract never reports G.T. (gasoline tank) symbols, so watch the vision model's gas-tank line. A "WARNING: reply was cut off" line means the model ran past its output limit; note that too.


In [ ]:
import os

os.environ["BLOCKHISTORY_FREE_MODEL"] = "qwen2.5vl:7b"
for tile in ["04858_12_1938-0050_r3_c2", "04858_12_1938-0050_r3_c0", "04858_12_1938-0050_r1_c2"]:
    sheet = tile.rsplit("_", 2)[0]
    print("#" * 20, tile)
    !python scripts/free_read_tile.py data/tiles/carondelet/1938/{sheet}/{tile}.jpg --vision

## 6. Save the results
Downloads the cached model outputs so you can copy them into `data/cache/free_vision/` on your laptop (they'll be reused for free), and write what the model got wrong in `docs/progress-log.md`.

In [ ]:
import os

from google.colab import files

if os.path.isdir("data/cache/free_vision"):
    !cd data/cache && zip -qr /content/free_vision_results.zip free_vision
    files.download("/content/free_vision_results.zip")
else:
    print("No vision results were saved; check the errors in step 5.")